# Part 10 · Notebook 05 — Purged cross-validation and feature importance

**Sessions:** S9–S10 (Cross-validation for financial ML · Feature importance) · [Lesson plan](../../docs/lessons/PART_10_MACHINE_LEARNING.md) · graded labs in [`labs/part10/`](../../labs/part10/)

**You will:**
1. Watch a shuffled k-fold find skill in pure noise.
2. Write the purging rule that removes overlapping training labels.
3. Compare MDI, MDA and single-feature importance on features with known importance.
4. Handle substitute features with clustered MDA.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure (a hidden regime, a known autocorrelation, planted importances, pure noise), so you can see what a model can learn, and that it learns nothing from noise.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p10lib.py is in notebooks/part10/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p10lib as p

p.use_course_style()

## 1. Skill in noise

`p.overlapping_dataset()` is a random walk: features are past returns and volatility, the label is whether the **next 20 days** go up. There is nothing to learn. But neighbouring rows share 19 of their 20 label days, so a shuffled k-fold puts near-copies of each test row in the training set.

**Purged k-fold** keeps contiguous test folds and drops every training label that overlaps the test period. A training row (label from `t0` to `t1`) survives only if it **ends before** the test fold starts (`t1 < test_start`) or **starts after** both the last test label ends and the embargo (`t0 > max(test_end, emb_end)`). Return the surviving positions (`np.where(...)[0]`).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyPurgedKFold(p.PurgedKFold):
    def split(self, X, y=None, groups=None):
        t0 = self.t1.index
        n, emb = len(t0), int(np.ceil(self.embargo * len(t0)))
        for test in np.array_split(np.arange(n), self.n_splits):
            test_start, test_end = t0[test[0]], self.t1.iloc[test].max()
            emb_end = t0[min(test[-1] + emb, n - 1)]
            train = ...                                              # ✍️
            yield train, test

X, y, t1 = p.overlapping_dataset()                       # NO signal; labels span the next 20 days
toy_t1 = pd.Series(pd.bdate_range("2024-01-01", periods=12)[[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 11, 11]],
                   index=pd.bdate_range("2024-01-01", periods=12))
cases = [(t1, X), (toy_t1, toy_t1), (t1, X, 3, 0.05)]
mine = [p.attempt(lambda t, data, k=5, e=0.01: list(MyPurgedKFold(t, k, e).split(data)), *c) for c in cases]
mine = p.check("PurgedKFold.split", mine, [list(p.PurgedKFold(c[0], *c[2:]).split(c[1])) for c in cases])
train, test = mine[0][2]
print(f"fold 3 of 5: {len(test)} test rows, {len(train)} training rows, {len(X) - len(test) - len(train)} purged or embargoed")

In [ ]:
from sklearn.model_selection import KFold

rf = p.make_model("rf")
schemes = {"shuffled k-fold": p.ShuffledKFold(5), "contiguous k-fold, no purging": KFold(5), "purged k-fold": p.PurgedKFold(t1, 5, 0.01)}
pd.Series({k: p.cv_scores(rf, X, y, cv)["auc"].mean() for k, cv in schemes.items()}, name="random forest AUC on pure noise").round(3).to_frame()

The shuffled k-fold reports an AUC near 0.7 on a random walk: pure leakage through overlapping labels. Contiguous folds leak only at their edges; purging removes even that. Every CV number in a financial ML report must come from purged folds (or a walk-forward).

## 2. Which features matter?

`p.planted_features()` knows the answer: `signal` drives the label, `twin` is a noisy copy of it, `noise_cont` and `noise_bin` are noise. **MDI** (the forest's built-in importance) is computed in sample and favours continuous features. **MDA** is out of sample: shuffle a feature in the test fold and measure how much the log loss gets worse. With `groups`, the columns of a group are shuffled **together** with one permutation (`Xs[cols].to_numpy()[perm]`).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from sklearn.base import clone
from sklearn.metrics import log_loss

def mda_importance(model, X, y, cv, n_repeats=3, seed=0, groups=None):
    rng = np.random.default_rng(seed)
    groups = groups or {c: [c] for c in X.columns}
    drops = {g: [] for g in groups}
    for tr, te in cv.split(X):
        m = clone(model).fit(X.iloc[tr], y.iloc[tr])
        base = log_loss(y.iloc[te], m.predict_proba(X.iloc[te]), labels=m.classes_)
        for g, cols in groups.items():
            for _ in range(n_repeats):
                Xs = X.iloc[te].copy()
                perm = rng.permutation(len(Xs))
                Xs[cols] = ...                                   # ✍️ the group's columns, rows permuted together
                drops[g].append(...)                             # ✍️ the log-loss increase over base
    return pd.Series({g: float(np.mean(v)) for g, v in drops.items()}).sort_values(ascending=False)

Xp, yp = p.planted_features()                            # y depends on `signal` only; `twin` = signal + noise
cv_p = p.PurgedKFold(pd.Series(Xp.index, index=Xp.index), 5, 0.0)   # independent rows: plain contiguous folds
rf = p.make_model("rf")
clusters = p.cluster_features(Xp)
mine = {"MDA": p.attempt(mda_importance, rf, Xp, yp, cv_p), "clustered MDA": p.attempt(mda_importance, rf, Xp, yp, cv_p, 3, 0, clusters)}
mine = p.check("mda_importance", mine, {"MDA": p.mda_importance(rf, Xp, yp, cv_p), "clustered MDA": p.mda_importance(rf, Xp, yp, cv_p, 3, 0, clusters)})
print("clusters:", clusters)

In [ ]:
mdi = p.mdi_importance(p.make_model("rf").fit(Xp, yp), Xp.columns)
sfi = p.sfi_importance(p.make_model("logit"), Xp, yp, cv_p)
pd.DataFrame({"MDI (in sample)": mdi, "MDA (log-loss increase)": mine["MDA"], "SFI (AUC alone)": sfi}).round(3)

MDI hands some importance to continuous noise, just because a tree can split on it. MDA gives noise about zero, and splits the signal's importance between `signal` and its `twin`: shuffling one hurts little because the other is still there (a substitution effect). Single-feature importance sees each feature alone, so `twin` looks almost as good as `signal`. **Clustered MDA** shuffles correlated features together: the `signal` cluster gets the full importance (compare the clustered value with the two single MDA values).

In [ ]:
mine["clustered MDA"].round(3).to_frame("clustered MDA")

## Wrap-up

* Overlapping labels + shuffled folds = skill in noise. Purge and embargo, always.
* Trust out-of-sample importance (MDA, SFI); MDI flatters continuous and noisy features.
* Correlated features share importance; cluster them before you drop "unimportant" ones.
* Graded version: `labs/part10/week35_validation` (`PurgedKFold`, `cv_scores`, `mda_importance`, `sfi_importance`, `cluster_features`).